# Daily Merge

Merge raw session CSV files into one `merged_<animal>.csv` file per animal, then optionally merge all animals into `merged_all_subjects.csv` for the full cohort of the selected line.

## 1. Setup

Run this cell first. It makes imports work whether the notebook is launched from the repo root or from inside `notebooks/ASD` or `notebooks/Stakes`.

In [5]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "analysis" / "daily_merge.py").exists() and (candidate / "DataFiles").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the Mafalda_analysis repo root from the current working directory."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/mafaldavalente/Documents/Mafalda_analysis')

## 2. Choose Dataset

Set `RAT = None` to process every animal in the cohort, or set it to one animal ID such as `"ASD0026"`.

In [19]:
LINE = "SHANK3"
COHORT = "cohort2"
RAT = None  # e.g. "ASD0026", or None for all animals

MODE = "both"  # "session", "animals", or "both"
MODEL_FILE = None

## 3. Optional Session Removals

Edit `SESSION_EDITS` before merging if a daily CSV should be removed entirely, or if only the bad tail/range of a session should be removed or marked repeated. File names must match the raw CSV names inside each animal folder.


In [14]:
# Optional per-animal cleanup rules applied while creating merged_<animal>.csv.
# These affect the merged output only; they do not edit the raw daily CSV files.
#
# Actions:
# - drop_entire_session: skip that raw CSV completely
# - drop_from_trial: remove trials with trial >= start_trial
# - drop_trial_range: remove trials from start_trial through end_trial, inclusive
# - drop_block: remove one or more block values from a session file
# - mark_repeated_from: keep rows but set repeated_trial = True from start_trial onward

SESSION_EDITS = {
    # Previous provisions from DailyMerge.py.
    "ASD0013": [
        {"file": "out_ASD0013_251014.csv", "action": "mark_repeated_from", "start_trial": 6690},
    ],

    # Previous ASD0018 provisions from DailyMerge.py.
    # Change action to "drop_from_trial" if you want these rows removed instead.
    "ASD0018": [
        {"file": "ASD0018_out_251014.csv", "action": "mark_repeated_from", "start_trial": 7370},
        {"file": "ASD0018_out_251015.csv", "action": "mark_repeated_from", "start_trial": 8000},
        {"file": "out_ASD0018_251028.csv", "action": "mark_repeated_from", "start_trial": 10900},
        {"file": "out_ASD0018_251127.csv", "action": "mark_repeated_from", "start_trial": 22250},
    ],

    "ASD0052": [
        {"file": "out_ASD0052_260707.csv", "action": "mark_repeated_from", "start_trial": 39640},
    ],

    "ASD0058": [
        {"file": "out_ASD0058_260617.csv", "action": "drop_block", "blocks": [1, 2, 3]},
    ],

    "ASD0040": [
        {"file": "out_ASD0040_260715.csv", "action": "mark_repeated_from", "start_trial": 65140},
    ],

    # Examples for ASD0019. Uncomment/edit the raw filenames and thresholds as needed.
    # "ASD0019": [
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_entire_session"},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_from_trial", "start_trial": 5000},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_trial_range", "start_trial": 1000, "end_trial": 1500},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_block", "block": 2},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_block", "blocks": [2, 3]},
    # ],
}

SESSION_EDITS


{'ASD0013': [{'file': 'out_ASD0013_251014.csv',
   'action': 'mark_repeated_from',
   'start_trial': 6690}],
 'ASD0018': [{'file': 'ASD0018_out_251014.csv',
   'action': 'mark_repeated_from',
   'start_trial': 7370},
  {'file': 'ASD0018_out_251015.csv',
   'action': 'mark_repeated_from',
   'start_trial': 8000},
  {'file': 'out_ASD0018_251028.csv',
   'action': 'mark_repeated_from',
   'start_trial': 10900},
  {'file': 'out_ASD0018_251127.csv',
   'action': 'mark_repeated_from',
   'start_trial': 22250}],
 'ASD0052': [{'file': 'out_ASD0052_260707.csv',
   'action': 'mark_repeated_from',
   'start_trial': 39640}],
 'ASD0058': [{'file': 'out_ASD0058_260617.csv',
   'action': 'drop_block',
   'blocks': [1, 2, 3]}],
 'ASD0040': [{'file': 'out_ASD0040_260715.csv',
   'action': 'mark_repeated_from',
   'start_trial': 65140}]}

## 4. Optional Bad RT Values

Use `RT_VALUE_EDITS` when task outcomes and abort labels are valid, but the recorded numeric `timed_rt` values should be ignored in RT analyses. These edits keep the trials and only set `timed_rt` to missing in the merged outputs.


In [15]:
# Numeric RT values to ignore while keeping trials for accuracy/choice/abort analyses.
# This only blanks timed_rt and adds rt_value_valid / rt_value_note columns.
# It does not change success, abort_type, choices, trial counts, or repeated_trial.

RT_VALUE_EDITS = [
    {
        "setup": 2,
        "start_date": "2026-06-13",
        "end_date": "2026-06-18",  # update if the setup-2 issue continues
        "date_col": "source_date",
        "setup_col": "box",
        "rt_col": "timed_rt",
        "reason": "setup 2 RT value recording issue",
    },

]

RT_VALUE_EDITS


[{'setup': 2,
  'start_date': '2026-06-13',
  'end_date': '2026-06-18',
  'date_col': 'source_date',
  'setup_col': 'box',
  'rt_col': 'timed_rt',
  'reason': 'setup 2 RT value recording issue'}]

## 5. Preview Animals

Check which animals will be processed before writing merged files.

In [20]:
import importlib
import analysis.daily_merge as DailyMerge
importlib.reload(DailyMerge)
from analysis.daily_merge import get_animals_for_cohort, get_base_dir

base_dir = get_base_dir(LINE, COHORT)
animals = get_animals_for_cohort(LINE, COHORT, rat=RAT)

print(f"Base directory: {base_dir}")
print(f"Animals ({len(animals)}): {animals}")

Base directory: /Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/SHANK3_cohort2
Animals (6): ['ASD0072', 'ASD0073', 'ASD0074', 'ASD0075', 'ASD0076', 'ASD0077']


## 6. Merge Daily Files Per Animal

This creates or updates `merged_<animal>.csv` files in the cohort data folder.

In [21]:
import importlib
import analysis.daily_merge as DailyMerge
importlib.reload(DailyMerge)
from analysis.daily_merge import merge_session_files

if MODE in ("session", "both"):
    merge_session_files(
        line=LINE,
        cohort=COHORT,
        rat=RAT,
        session_edits=SESSION_EDITS,
        rt_value_edits=RT_VALUE_EDITS,
    )
else:
    print("Skipping per-animal session merge.")

Processing 6 animal(s) for SHANK3 cohort2: ASD0072, ASD0073, ASD0074, ASD0075, ASD0076, ASD0077
Using latest file 'out_ASD0072_260717.csv' as column reference (86 columns).
Total unique columns across all files: 86
✅ Added out_ASD0072_260617.csv (26 rows)
✅ Added out_ASD0072_260618.csv (42 rows)
✅ Added out_ASD0072_260622.csv (19 rows)
✅ Added out_ASD0072_260623.csv (28 rows)
✅ Added out_ASD0072_260624.csv (19 rows)
✅ Added out_ASD0072_260625.csv (578 rows)
✅ Added out_ASD0072_260626.csv (530 rows)
✅ Added out_ASD0072_260629.csv (647 rows)
✅ Added out_ASD0072_260701.csv (527 rows)
✅ Added out_ASD0072_260702.csv (638 rows)
✅ Added out_ASD0072_260703.csv (516 rows)
✅ Added out_ASD0072_260707.csv (397 rows)
✅ Added out_ASD0072_260708.csv (385 rows)
✅ Added out_ASD0072_260709.csv (355 rows)
✅ Added out_ASD0072_260713.csv (298 rows)
✅ Added out_ASD0072_260716.csv (371 rows)
✅ Added out_ASD0072_260717.csv (767 rows)
🧹 Set timed_rt=NaN for 0 rows from 2026-06-13 to 2026-06-18 on setup 2
🎉 Mer

## 7. Merge Animals Into Cohort File

This creates or updates `merged_all_subjects.csv` in the cohort data folder.

In [22]:
import importlib
import analysis.daily_merge as DailyMerge
importlib.reload(DailyMerge)
from analysis.daily_merge import merge_subject_files_with_model

if MODE in ("animals", "both"):
    merged_df = merge_subject_files_with_model(
        line=LINE,
        cohort=COHORT,
        model_file=MODEL_FILE,
    )
else:
    merged_df = None
    print("Skipping cohort-level animal merge.")

if merged_df is not None:
    display(merged_df.head())
    print(merged_df.shape)

📘 Using model file 'merged_ASD0072.csv' with 92 columns.
🧾 Total unique columns across all subjects: 92
✅ Added merged_ASD0072.csv (6143 rows)
✅ Added merged_ASD0073.csv (6343 rows)
✅ Added merged_ASD0074.csv (5723 rows)
✅ Added merged_ASD0075.csv (6815 rows)
✅ Added merged_ASD0076.csv (6453 rows)
✅ Added merged_ASD0077.csv (4822 rows)
🎉 Saved merged dataset: /Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/SHANK3_cohort2/merged_all_subjects.csv
Final shape: (36299, 92)


,animal,batch,experimenter,version,bias,repeated_trial,trial,trial_start,tared_trial_start,trial_end,...,rt_start_frame,mt_start_frame,lnp_start_frame,lnp_end_frame,stim_dur,stim_dur_label,source_file,source_date,rt_value_valid,rt_value_note
0,ASD0072,shank3,HY,0.10.0,0.00,True,1,3.864549e+09,0.000000,3.864549e+09,...,NaN,NaN,NaN,NaN,6000,RT,out_ASD0072_260617.csv,2026-06-17,True,NaN
1,ASD0072,shank3,HY,0.10.0,0.00,True,2,3.864549e+09,12.192000,3.864550e+09,...,NaN,NaN,NaN,NaN,6000,RT,out_ASD0072_260617.csv,2026-06-17,True,NaN
2,ASD0072,shank3,HY,0.10.0,0.00,True,3,3.864550e+09,194.216000,3.864550e+09,...,NaN,NaN,NaN,NaN,6000,RT,out_ASD0072_260617.csv,2026-06-17,True,NaN
3,ASD0072,shank3,HY,0.10.0,0.00,True,4,3.864550e+09,376.225024,3.864550e+09,...,NaN,NaN,NaN,NaN,6000,RT,out_ASD0072_260617.csv,2026-06-17,True,NaN
4,ASD0072,shank3,HY,0.10.0,-0.04,True,5,3.864550e+09,558.252000,3.864550e+09,...,3.0,49.0,NaN,NaN,6000,RT,out_ASD0072_260617.csv,2026-06-17,True,NaN


(36299, 92)
